# SRL-NER Inference S3.2 (Winner) → Seluruh Sirah

Inference model winner **S3.2-scl-aug-iter4** (TEST F1 entity = 0.9537) ke seluruh `sirah_chunks_final.csv` (1094 chunks). Output untuk regenerate Knowledge Graph v3.

**Prerequisite di Drive:**
- `MyDrive/TA-Sirah/` punya `sirah_chunks_final.csv`
- `MyDrive/TA-Sirah/output/models/bert-only-sirah-ner-S3-2-scl-aug-v2-0.9-iteration-4/` (model weights, hasil run S3.2 sebelumnya)

**Output:**
- `sirah_predicted_v3_token.csv` — token-level (mirror format `test.csv`/manual)
- `sirah_predicted_v3_entity.csv` — entity-level (untuk relation_extraction)
- `inference_runtime.json`

In [ ]:
!pip install -q -U transformers accelerate seaborn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import json
import time
import datetime
from pathlib import Path

import pandas as pd
import torch
from tqdm import tqdm
from transformers import pipeline

DRIVE_ROOT = Path('/content/drive/MyDrive/TA-Sirah')
MODEL_DIR = DRIVE_ROOT / 'output' / 'models' / 'bert-only-sirah-ner-S3-2-scl-aug-v2-0.9-iteration-4'
CHUNKS_CSV = DRIVE_ROOT / 'sirah_chunks_final.csv'

OUT_DIR = DRIVE_ROOT / 'output' / 'inference_v3'
OUT_DIR.mkdir(parents=True, exist_ok=True)

TOKEN_CSV = OUT_DIR / 'sirah_predicted_v3_token.csv'
ENTITY_CSV = OUT_DIR / 'sirah_predicted_v3_entity.csv'
RUNTIME_JSON = OUT_DIR / 'inference_runtime.json'

print('CUDA available :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU            :', torch.cuda.get_device_name(0))
print('Model dir      :', MODEL_DIR, '->', 'OK' if MODEL_DIR.exists() else 'MISSING')
print('Chunks CSV     :', CHUNKS_CSV, '->', 'OK' if CHUNKS_CSV.exists() else 'MISSING')

In [ ]:
df_chunks = pd.read_csv(CHUNKS_CSV, sep=';', encoding='utf-8-sig')
df_chunks['teks_chunk'] = df_chunks['teks_chunk'].fillna('').astype(str)
print(f'Total chunks: {len(df_chunks)}')
df_chunks.head(2)

In [ ]:
device = 0 if torch.cuda.is_available() else -1
ner = pipeline(
    'token-classification',
    model=str(MODEL_DIR),
    aggregation_strategy='first',
    device=device,
)
print('NER pipeline loaded (aggregation_strategy=first)')
print('Note: aggregation=first menghindari sub-word fragmentation yang muncul di run "simple" sebelumnya.')

## Inference + dual-output (token-level + entity-level)

Untuk tiap chunk:
1. Whitespace tokenize teks
2. Run NER pipeline → entity-level predictions
3. Map back ke token-level BIO labels (sejalan dengan `extract_entities_from_result` di S3.2 notebook)
4. Append ke 2 output CSV

In [ ]:
def extract_entities_from_result(tokens, result):
    """Map HF pipeline result -> token-level BIO labels."""
    predicted = []
    current_index = 0
    prev_span_id = None
    for token in tokens:
        hit = None
        for idx, entry in enumerate(result):
            if entry['start'] <= current_index < entry['end']:
                hit = idx
                break
        if hit is None:
            label = 'O'
            prev_span_id = None
        else:
            group = result[hit]['entity_group']
            if group.startswith(('B_', 'I_')):
                label = group
            else:
                label = f'B_{group}' if hit != prev_span_id else f'I_{group}'
            prev_span_id = hit
        predicted.append(label)
        current_index += len(token) + 1
    return predicted


def run_inference(df_chunks):
    """Run NER on all chunks, return (token_rows, entity_rows)."""
    token_rows = []
    entity_rows = []
    for _, row in tqdm(df_chunks.iterrows(), total=len(df_chunks), desc='inference'):
        chunk_id = row['chunk_id']
        halaman = row['halaman']
        text = row['teks_chunk']
        if not text.strip():
            continue
        tokens = text.split()
        try:
            result = ner(text)
        except Exception as e:
            print(f'[WARN] {chunk_id} error: {e}')
            result = []
        # Token-level
        labels = extract_entities_from_result(tokens, result)
        for idx, (tok, lab) in enumerate(zip(tokens, labels), start=1):
            token_rows.append({
                'text_id': chunk_id,
                'id': f'{chunk_id}.{idx:03d}',
                'token': tok,
                'predicted_label': lab,
                'halaman': halaman,
            })
        # Entity-level (one row per entity span)
        for ent in result:
            entity_rows.append({
                'chunk_id': chunk_id,
                'halaman': halaman,
                'entity_text': ent['word'],
                'entity_label': ent['entity_group'].replace('B_', '').replace('I_', ''),
                'start_char': ent['start'],
                'end_char': ent['end'],
                'confidence': float(ent['score']),
            })
    return token_rows, entity_rows


t_start = time.time()
iso_start = datetime.datetime.now().isoformat(timespec='seconds')
token_rows, entity_rows = run_inference(df_chunks)
t_end = time.time()
iso_end = datetime.datetime.now().isoformat(timespec='seconds')

print(f'\nInference selesai dalam {(t_end - t_start):.1f} detik')

In [ ]:
df_token = pd.DataFrame(token_rows, columns=['text_id', 'id', 'token', 'predicted_label', 'halaman'])
df_entity = pd.DataFrame(entity_rows, columns=['chunk_id', 'halaman', 'entity_text', 'entity_label', 'start_char', 'end_char', 'confidence'])

df_token.to_csv(TOKEN_CSV, index=False, encoding='utf-8-sig')
df_entity.to_csv(ENTITY_CSV, index=False, sep=';', encoding='utf-8-sig')

runtime_log = {
    'model_dir': str(MODEL_DIR),
    'started_at': iso_start,
    'ended_at': iso_end,
    'total_seconds': round(t_end - t_start, 2),
    'total_minutes': round((t_end - t_start) / 60, 2),
    'n_chunks': int(len(df_chunks)),
    'n_chunks_predicted': int(df_token['text_id'].nunique()),
    'n_tokens': int(len(df_token)),
    'n_entities': int(len(df_entity)),
}
RUNTIME_JSON.write_text(json.dumps(runtime_log, indent=2, ensure_ascii=False))

print(f'\nOutput files:')
print(f'  {TOKEN_CSV}  ({len(df_token)} rows)')
print(f'  {ENTITY_CSV}  ({len(df_entity)} rows)')
print(f'  {RUNTIME_JSON}')
print('\nLabel distribution (token-level):')
print(df_token['predicted_label'].value_counts().to_string())
print('\nEntity label distribution:')
print(df_entity['entity_label'].value_counts().to_string())

## Selesai

Download 3 file di `MyDrive/TA-Sirah/output/inference_v3/` ke repo lokal:
- `sirah_predicted_v3_token.csv` → `data/result/pseudo-labelling/SRL-NER/sirah_predicted_v3_token.csv`
- `sirah_predicted_v3_entity.csv` → `data/result/pseudo-labelling/SRL-NER/sirah_predicted_v3_entity.csv`
- `inference_runtime.json` → idem

Lalu jalankan script `regenerate_kg_v3.py` (akan dibuat di sesi berikutnya) untuk:
1. Build nodes_v3.csv + edges_v3.csv dari entity prediction
2. Apply event_period mapping
3. Generate import_sirah_v3.cypher untuk Neo4j